# 02 · Необходимость, число прогонов и честность метрики**Ничего не нужно: ни ключа, ни модели, ни сети.** Всё считается на ужепрогнанных данных:- `data/full_cf_stopwords_results.csv` — 468 контрфактуальных запусков- `data/benchmark_results.csv` — 300 промптов, по одному прогону на каждый,  с полным текстом ответовПрогоны стоили времени и вызовов модели один раз. Результат сохранён целикомименно затем, чтобы переанализировать его сколько угодно раз бесплатно.Три сюжета:1. Как меряется **необходимость** и чем она отличается от того, что выдаёт атрибуция.2. Сколько нужно **прогонов**, чтобы порядок важностей вообще был определён.3. Почему **метрика врёт сильнее агента** — и как это поймать.

In [ ]:
import sys; sys.path.insert(0, "..")import numpy as npimport pandas as pdfrom sanity_kit import bootstrap_order_prob, min_repeats, wilson_cifrom agent.benchmark import (load_runs, match_analysis, tool_frequency_summary,                             classify_mismatches)from agent.tools import TOOL_NAMESpd.set_option("display.precision", 4)

## 1. Реальные контрфактуалы`flipped = True` значит: убрали слово из промпта, и агент перестал вызыватьинструмент. Это прямое измерение необходимости — вмешательством, а не оценкой.

In [ ]:
cf = pd.read_csv("../data/full_cf_stopwords_results.csv")print(cf.shape, "\n")print(cf.head(3).to_string(index=False), "\n")pn = cf.groupby("tool")["flipped"].agg(["sum", "count"])pn["PN"] = pn["sum"] / pn["count"]pn = pn.sort_values("PN", ascending=False)for tool, row in pn.iterrows():    lo, hi = wilson_ci(int(row["sum"]), int(row["count"]))    print(f"{tool:<18} PN={row['PN']:6.1%}  [{lo:5.1%}, {hi:5.1%}]  n={int(row['count'])}")

## 2. Сколько прогонов надоНаивный эксперимент — три прогона на инструмент. Посмотрим, что при этомможно утверждать.Ничья — отдельная категория, а не «почти угадали»: это означает, что данныене определяют порядок вообще.

In [ ]:
rates = {t: float(r["PN"]) for t, r in pn.iterrows()}rates = {k: min(max(v, 1e-4), 1 - 1e-4) for k, v in rates.items()}print("истинные частоты:", {k: round(v, 3) for k, v in rates.items()}, "\n")print(f"{'n на инструмент':>16} {'верный порядок':>16} {'ничья':>10} {'неверный':>10}")for n_per in (3, 5, 10, 20, 50, 100, 300):    r = bootstrap_order_prob(rates, n_per, n_boot=50_000)    print(f"{n_per:>16} {r['p_correct']:>15.1%} {r['p_tie']:>10.1%} {r['p_wrong']:>10.1%}")

In [ ]:
print("Сколько прогонов нужно, чтобы отличить соседние пары:\n")ordered = list(rates.items())for (t1, p1), (t2, p2) in zip(ordered, ordered[1:]):    try:        print(f"  {t1} vs {t2}: {min_repeats(p1, p2)} прогонов на инструмент")    except ValueError as e:        print(f"  {t1} vs {t2}: {e}")print("\nНоль срабатываний из N — какой это на самом деле интервал:\n")for n in (1, 3, 5, 10, 30, 100):    lo, hi = wilson_ci(0, n)    print(f"  0 из {n:>3}:  [{lo:.1%}, {hi:5.1%}]")

## 3. Метрика врёт сильнее агента300 промптов, сгенерированных по шаблонам: каждый инструмент нужен ровнов 130 из них и не нужен в 170. Для каждого промпта записано, какиеинструменты ожидались и какие агент вызвал.Вопрос простой: как часто агент прав? Ответ зависит от того, как написаносравнение — и зависит катастрофически.

In [ ]:
runs = load_runs()print(runs.shape)print(runs.head(3).to_string(index=False, max_colwidth=40))

In [ ]:
m = match_analysis(runs)print(m)

Сорок семь процентов против девяноста восьми. Разница целиком в одной строкекода: `"a+b" == "b+a"` ложно, хотя это один и тот же набор инструментов.Если бы вы отчитывались первым числом, вы бы назвали агента сломанными пошли его чинить. Чинить надо было сравнение.Это не редкий случай. Ровно такое сравнение вы напишете, когда будетепроверять, совпал ли набор сработавших правил с ожидаемым.

In [ ]:
# Настоящие расхождения: их пять, и они распадаются на два разных типаreal = classify_mismatches(m.mismatches)print(real.to_string(index=False, max_colwidth=44))print()print(real["тип"].value_counts().to_string())

Три из пяти — `spurious_call`: инструмент не нужен, а агент его вызвал.«What is the capital of Egypt?» → `get_news`. «How many bones are in thehuman body?» → `calculate`.Запомните эти промпты: в ноутбуке `03` на том же семействе no-tool промптовAgentSHAP выдаёт ненулевые вклады. Аксиома dummy нарушается дважды — сначалав поведении агента, потом в его объяснении, и это независимые поломки.## Recall и FPR по инструментамТеперь, когда сравнение написано правильно, числа можно считать.

In [ ]:
summary = tool_frequency_summary(runs, TOOL_NAMES)print(summary.to_string())print("\nДоверительные интервалы для FPR (иначе 0.0059 выглядит как точное число):\n")for tool, row in summary.iterrows():    k = int(round(row["false_positive_rate"] * row["n_not_expected"]))    lo, hi = wilson_ci(k, int(row["n_not_expected"]))    print(f"  {tool:<18} FPR={row['false_positive_rate']:.4f}  "          f"[{lo:.4f}, {hi:.4f}]  ({k} из {int(row['n_not_expected'])})")

Обратите внимание: FPR у `get_weather` и `get_stock_price` равен нулю,но интервал — не ноль, а примерно [0, 0.022]. Ноль из 170 не означает«никогда»; он означает «реже, чем мы можем различить на 170 наблюдениях».Именно так и надо отчитываться о защите, которая ни разу не ошибласьна вашем тестовом наборе.## Чеклист для хакатонаЕсли вы заявляете «наш сенсор детектит эту атаку» или «вот это правилосработало, поэтому мы заблокировали»:1. **Сколько прогонов** за этим утверждением? Один — не измерение.2. **Какой интервал?** Ноль срабатываний из трёх — это не 0 %, а [0 %, 56 %].3. **Правильно ли написано сравнение?** 47 % против 98 % на одних и тех же   данных — это не про агента, это про метрику.4. **Проверяли контрфактуал** — без этого звена утечка бы случилась?5. **Нет ли избыточности?** Если два правила всегда срабатывают вместе,   важность каждого не определена ни при каком объёме данных.`sanity_kit` отвечает на 1, 2, 4 и 5. Третий пункт — ваша ответственность,и он же самый дешёвый в исправлении и самый дорогой в последствиях.